# Lesson 10 - AI Agents in Production

In this lesson you will learn **production patterns** for AI agents using the Microsoft Agent Framework (Python). We cover:

- **Observability** — adding timing and logging to agent interactions
- **Evaluation** — using an evaluator agent to score response quality
- **Cost management** — strategies for token optimization and model selection

The scenario is a **travel agent** that helps users plan trips, with monitoring and evaluation layered on top.

## Setup

In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [1]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

c:\Users\lujan\Proyectos\Cursos\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\lujan\Proyectos\Cursos\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


In [2]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Production Considerations

Moving AI agents from prototypes to production requires careful attention to three pillars:

1. **Observability** — You need visibility into what the agent is doing, how long it takes, and which tools it calls. Without tracing and logging, debugging production issues is nearly impossible.

2. **Evaluation** — Automated quality checks ensure the agent's responses remain accurate, complete, and helpful over time. An evaluator agent can score responses against defined criteria.

3. **Cost Management** — Token usage directly impacts cost. Strategies like prompt optimization, model selection, and caching help keep expenses under control without sacrificing quality.

## Building an Observable Agent

We define travel tools and wrap the agent call with timing so we can monitor latency. In production you would integrate with OpenTelemetry or a similar tracing backend.

In [3]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [4]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

Response (26.54s):
Great — I can plan a compact, enjoyable day in Paris and pull flight options once I have your origin and dates. First, activity recommendations (I pulled these for Paris): Louvre Museum, Eiffel Tower, Seine river cruise, Montmartre walking tour.

Before I search flights I need:
- Your departure city/airport
- Travel date(s) and preferred departure/return times (same day return or overnight?)
- Budget or cabin class preference (economy/premium)
- Any airline or routing preferences (direct only? flexible times?)

While you decide, here’s a ready-to-use day plan with logistics and practical tips:

Suggested one-day Paris itinerary (timed for a full day arrival)

- 08:30–11:00 — Louvre Museum
  - Arrive at opening (usually 9:00; check exact day/time). Prebook a timed-entry ticket to skip lines.
  - Highlights: Denon wing (Mona Lisa, Winged Victory, Venus de Milo). Plan ~2–2.5 hours if you want a focused visit.
  - Metro: Palais Royal – Musée du Louvre (line 1) or Tuileri

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time

In [5]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

Evaluation:
Completeness: 4/5
- Strong coverage of activities with a detailed, timed one-day itinerary and practical logistics. The flights section gives useful general guidance and requests the necessary info, but it doesn’t present concrete flight options yet.

Accuracy: 5/5
- Information is consistent and factually correct (museum hours caveat, metro stations, cruise/booking suggestions, luggage/safety tips, approximate pricing). No obvious errors.

Helpfulness: 5/5
- Very actionable for a traveler: timed schedule, transport tips, booking priorities, luggage advice, and clear next steps for the agent to pull flights and refine timing.

Overall Score: 5/5
- High-quality, professional response that balances a ready-to-use day plan with the right follow-up questions to deliver tailored flight options and final bookings.


## Cost Management Strategies

Controlling costs is critical for production AI agents. Here are key strategies:

| Strategy | Description |
|---|---|
| **Prompt optimization** | Keep system instructions concise. Remove redundant context to reduce input tokens. |
| **Model selection** | Use smaller, cheaper models (e.g. GPT-4o-mini) for simple tasks like classification or extraction, and reserve larger models for complex reasoning. |
| **Caching** | Cache tool results and frequent queries to avoid redundant API calls. |
| **Token budgets** | Set `max_tokens` limits to prevent unexpectedly long responses. |
| **Batching** | Group multiple user queries into a single API call where possible. |

In practice, a tiered approach works well: route straightforward requests to a fast, inexpensive model and escalate only complex queries to a more capable one.

## Summary

In this lesson you learned how to:

1. **Add observability** to agent interactions with timing and logging, laying the groundwork for tracing and monitoring.
2. **Evaluate agent responses** automatically using an evaluator agent that scores completeness, accuracy, and helpfulness.
3. **Manage costs** through prompt optimization, model selection, caching, and token budgets.

These production patterns help ensure your AI agents are reliable, measurable, and cost-effective at scale.